In [5]:
import torch 
import torch.nn as nn 
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder # Stream data from images stored in folders

import os 
from PIL import Image 
from collections import Counter # to count of unique items in an iterable 

import urllib.request
import shutil
from urllib.parse import urlparse


### 1. DataLoader for CV
### Dogs vs Cats Dataset

In [ ]:
DATA_ROOT_FOLDER = "../../data"
IMAGE_FOLDER_ROOT = os.path.join(DATA_ROOT_FOLDER,"PetImages")

def downloadData(url):
    filename = os.path.basename(urlparse(url).path)
    output_path = os.path.join(DATA_ROOT_FOLDER, filename)

    urllib.request.urlretrieve(url, output_path) # download file 
    shutil.unpack_archive(output_path, DATA_ROOT_FOLDER) # extract 

downloadData("https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip")

### Component of a Dataset 
- init: Initialize the model class with everything you want to store as class variables 
- len: to tell the Dataset how many samples there are. When we go to grab a sample, it can grab all the samples from index 0 to index len 
- getitem: given some index between 0 and the length of data, it will grab some sample

In [ ]:
class DogsVsCatsDatasets(Dataset):
    def __init__(self, path_to_folder):
        path_to_cats = os.path.join(path_to_folder, "Cat")
        path_to_dogs = os.path.join(path_to_folder, "Dog")

        cat_files = os.listdir(path_to_cats)
        dog_files = os.listdir(path_to_dogs)

        path_to_cat_files = [os.path.join(path_to_cats, filename) for filename in cat_files]
        path_to_dog_files = [os.path.join(path_to_dogs, filename) for filename in dog_files]

        # save in memory the path to the images => when needed the data => load it to memory instead of load the entire data into memory
        self.training_files = path_to_cat_files + path_to_dog_files
        
        self.dog_label = 0 
        self.cat_label = 1

        # transforms.ToTensor(): 
        # - convert to tensor, 
        # - change (H, W, C) => (C, H, W) as pytorch requires input (batch_size, channels, height, width)
        # - Scale 0–255 → 0–1 = pixel_value/255 
        # - convert dtype from uint8 → float32
        self.transform = transforms.ToTensor() 

    def __len__(self):
        return len(self.training_files) 

    def __getitem__(self, idx):
        path_to_image = self.training_files[idx]
        label = self.dog_label if "Dog" in path_to_image else self.cat_label
        image = Image.open(path_to_image).convert("RGB") # for sure every image is converted to RGB if they're not, .convert("L"): to gray
        # print(image) # is a JpegImage, and size 500x375, RGB

        # print(np.array(image)) # shape: (375, 500, 3), values in [0, 255]
        image = self.transform(image) # automatically divided by 255 
        # print(image) # shape: [3, 375, 500], value, values in [0, 1]

        return image, label

dataset = DogsVsCatsDatasets(IMAGE_FOLDER_ROOT)

for sample in dataset: # call __getitem___
    break

### DataLoader
Sample data in minibatches from dataset, with collate_fn=default_collate
- stack all the samples in a batch into a tensor: torch.stack([img1, img2, ...])
- It works only if all samples have the same size => we can use ReSize() or use a custom collate_fn

Use a custom collate_fn: 
- when samples have different sizes: need to resize, padding,... or return tuple instead of a single tensor 
- when data is complex => need customs to make batch 
- when data augmentation/preprocessing/special transforms

In [ ]:
BATCH_SIZE= 16 

dataloader = DataLoader(dataset=dataset, batch_size=16, shuffle=True)

for images, labels in dataloader: 
    print(images.shape)
    print(labels)
    break

### Torchvision Transforms
- ToTensor(): 
    - convert to tensor, 
    - transpose (H, W, C) => (C, H, W) as pytorch requires input (batch_size, channels, height, width)
    - Scale 0–255 → 0–1 = pixel_value/255 
    - convert dtype from uint8 → float32
- Resize(): 
- Normalize(): Normalize mean/std of the image on each channel
- RandomHorizontalFlip(): Randomly flips an image horizontally with soem probability (default 0.5)
- RandomVerticalFlip(): Randomly flips an image vertically with probability (default 0.5)


- Compose(): to stick together multiple transformations in a single sequence

In [ ]:
# add a composition of transforms and add it to the datasets
img_transforms = transforms.Compose([
    transforms.Resize((224, 224)), # a standard shape that we pass into models like ResNet, ImageNet
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406], # Precomputed values from the ImageNet dataset (R, G, B)
                        std  = [0.229, 0.224, 0.225])
])

class DogsVsCatsDatasets(Dataset):
    def __init__(self, path_to_folder):
        path_to_cats = os.path.join(path_to_folder, "Cat")
        path_to_dogs = os.path.join(path_to_folder, "Dog")

        cat_files = os.listdir(path_to_cats)
        dog_files = os.listdir(path_to_dogs)

        path_to_cat_files = [os.path.join(path_to_cats, filename) for filename in cat_files]
        path_to_dog_files = [os.path.join(path_to_dogs, filename) for filename in dog_files]

        # save in memory the path to the images => when needed the data => load it to memory instead of load the entire data into memory
        self.training_files = path_to_cat_files + path_to_dog_files
        
        self.dog_label = 0 
        self.cat_label = 1

        # compose of transforms
        self.transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.ToTensor(),
                transforms.Normalize(mean = [0.485, 0.456, 0.406], # Precomputed values from the ImageNet dataset (R, G, B)
                                    std  = [0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.training_files) 

    def __getitem__(self, idx):
        path_to_image = self.training_files[idx]
        label = self.dog_label if "Dog" in path_to_image else self.cat_label
        image = Image.open(path_to_image).convert("RGB") # for sure every image is converted to RGB if they're not, .convert("L"): to gray
        # print(image) # is a JpegImage, and size 500x375, RGB

        # print(np.array(image)) # shape: (375, 500, 3), values in [0, 255]
        image = self.transform(image) # automatically divided by 255 
        # print(image) # shape: [3, 375, 500], value, values in [0, 1]

        return image, label

dataset = DogsVsCatsDatasets(IMAGE_FOLDER_ROOT)
dataloader = DataLoader(dataset=dataset, batch_size=16, shuffle=True)

for images, labels in dataloader: 
    print(images.shape)
    print(labels) 
    break

(tensor([[[ 1.3584,  1.3927,  1.4440,  ...,  2.0605,  2.0092,  1.9749],
         [ 1.3584,  1.3927,  1.4440,  ...,  2.0434,  2.0092,  1.9920],
         [ 1.3584,  1.3927,  1.4440,  ...,  2.0434,  2.0263,  1.9920],
         ...,
         [ 0.5193,  0.5364,  0.5536,  ..., -2.0665, -2.0665, -2.0665],
         [ 0.5022,  0.5022,  0.5193,  ..., -2.0837, -2.0837, -2.0837],
         [ 0.4679,  0.4851,  0.5022,  ..., -2.1008, -2.1008, -2.1008]],

        [[ 0.8354,  0.8704,  0.9230,  ...,  1.5182,  1.5007,  1.4657],
         [ 0.8354,  0.8704,  0.9230,  ...,  1.5532,  1.5007,  1.4832],
         [ 0.8354,  0.8704,  0.9230,  ...,  1.5532,  1.5182,  1.4832],
         ...,
         [ 0.1176,  0.1352,  0.1527,  ..., -1.9832, -1.9832, -1.9832],
         [ 0.1001,  0.1001,  0.1176,  ..., -2.0007, -2.0007, -2.0007],
         [ 0.0651,  0.0826,  0.1001,  ..., -2.0182, -2.0182, -2.0182]],

        [[-0.2881, -0.2532, -0.2010,  ...,  0.3045,  0.3219,  0.3045],
         [-0.2881, -0.2532, -0.2010,  ...,  

### Random Split

In [ ]:
NUM_WORKERS = 2

num_train_samples = int(0.9*len(dataset)) # 90% of data is for training 
num_test_samples = len(dataset) - num_train_samples

train_dataset, test_dataset = torch.utils.data.random_split(dataset, lengths=[num_train_samples, num_test_samples])

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

### ImageFolder
the built-in class that does the same things as in DogsVsCatsDatasets

In [31]:
ds = ImageFolder(root=IMAGE_FOLDER_ROOT)
print(ds)
print(ds.classes)

Dataset ImageFolder
    Number of datapoints: 25000
    Root location: ../data/PetImages/
['Cat', 'Dog']


### 2. DataLoaders for NLP

### IMDB Dataset

In [7]:
downloadData("http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz")

### Tokenizer

In [8]:
from tqdm.notebook import tqdm

PATH_TO_IMDB_DATA = os.path.join(DATA_ROOT_FOLDER, "aclImdb", "train")

pos_folder = os.path.join(PATH_TO_IMDB_DATA, "pos")
neg_folder = os.path.join(PATH_TO_IMDB_DATA, "neg")

path_to_pos_txt = [os.path.join(pos_folder, file) for file in os.listdir(pos_folder)]
path_to_neg_txt = [os.path.join(neg_folder, file) for file in os.listdir(neg_folder)]

training_files = path_to_pos_txt + path_to_neg_txt

alltxt = ""
for file in tqdm(training_files): 
    with open(file, 'r', encoding="utf-8") as f: 
        txt = f.readlines()[0]
        alltxt += txt

  0%|          | 0/25000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [47]:
unique_counts = dict(Counter(alltxt)) # the number of occurrences of each string 
characters = [key for (key, value) in unique_counts.items() if value != 1500] # remove characters that the numbers of occurrences < a threshold
characters = sorted(characters)
characters.append("<UNK>")
characters.append("<PAD>") # if sample sizes are not the same -> need to pad to the shortest txt to the longest 

char2idx = {}
idx2char = {}

for idx, char in enumerate(characters): 
    char2idx[char] = idx
    idx2char[idx] = char 


In [48]:
char2idx

{'\x08': 0,
 '\t': 1,
 '\x10': 2,
 ' ': 3,
 '!': 4,
 '"': 5,
 '#': 6,
 '$': 7,
 '%': 8,
 '&': 9,
 "'": 10,
 '(': 11,
 ')': 12,
 '*': 13,
 '+': 14,
 ',': 15,
 '-': 16,
 '.': 17,
 '/': 18,
 '0': 19,
 '1': 20,
 '2': 21,
 '3': 22,
 '4': 23,
 '5': 24,
 '6': 25,
 '7': 26,
 '8': 27,
 '9': 28,
 ':': 29,
 ';': 30,
 '<': 31,
 '=': 32,
 '>': 33,
 '?': 34,
 '@': 35,
 'A': 36,
 'B': 37,
 'C': 38,
 'D': 39,
 'E': 40,
 'F': 41,
 'G': 42,
 'H': 43,
 'I': 44,
 'J': 45,
 'K': 46,
 'L': 47,
 'M': 48,
 'N': 49,
 'O': 50,
 'P': 51,
 'Q': 52,
 'R': 53,
 'S': 54,
 'T': 55,
 'U': 56,
 'V': 57,
 'W': 58,
 'X': 59,
 'Y': 60,
 'Z': 61,
 '[': 62,
 '\\': 63,
 ']': 64,
 '^': 65,
 '_': 66,
 '`': 67,
 'a': 68,
 'b': 69,
 'c': 70,
 'd': 71,
 'e': 72,
 'f': 73,
 'g': 74,
 'h': 75,
 'i': 76,
 'j': 77,
 'k': 78,
 'l': 79,
 'm': 80,
 'n': 81,
 'o': 82,
 'p': 83,
 'q': 84,
 'r': 85,
 's': 86,
 't': 87,
 'u': 88,
 'v': 89,
 'w': 90,
 'x': 91,
 'y': 92,
 'z': 93,
 '{': 94,
 '|': 95,
 '}': 96,
 '~': 97,
 '\x80': 98,
 '\x84': 

In [ ]:
class IMDBDataset(Dataset): 
    def __init__(self, path_to_data, char2idx):
        pos_folder = os.path.join(path_to_data, "pos")
        neg_folder = os.path.join(path_to_data, "neg")

        path_to_pos_txt = [os.path.join(pos_folder, file) for file in os.listdir(pos_folder)]
        path_to_neg_txt = [os.path.join(neg_folder, file) for file in os.listdir(neg_folder)]

        self.training_files = path_to_pos_txt + path_to_neg_txt 
        self.tokenizer = char2idx

        self.pos_label = 1
        self.neg_label = 0
    
    def __len__(self):
        return len(self.training_files) 

    def __getitem__(self, idx): 
        path_to_txt = self.training_files[idx] 

        with open(path_to_txt, 'r', encoding='utf-8') as f: 
            txt = f.readlines()[0]

        tokenized = []
        
        for char in txt: 
            if char in self.tokenizer.keys(): 
                tokenized.append(self.tokenizer[char])
            else: 
                tokenized.append(self.tokenizer["<UNK>"])

        sample = torch.tensor(tokenized)
        label = self.pos_label if "pos" in path_to_txt else self.neg_label

        return sample,label

PATH_TO_IMDB_DATA = os.path.join(DATA_ROOT_FOLDER, "aclImdb", "train")
dataset = IMDBDataset(PATH_TO_IMDB_DATA, char2idx)

for sample, lbl in dataset: 
    print(sample, lbl)
    break

tensor([37, 85, 82, 80, 90, 72, 79, 79,  3, 43, 76, 74, 75,  3, 76, 86,  3, 68,
         3, 70, 68, 85, 87, 82, 82, 81,  3, 70, 82, 80, 72, 71, 92, 17,  3, 44,
        87,  3, 85, 68, 81,  3, 68, 87,  3, 87, 75, 72,  3, 86, 68, 80, 72,  3,
        87, 76, 80, 72,  3, 68, 86,  3, 86, 82, 80, 72,  3, 82, 87, 75, 72, 85,
         3, 83, 85, 82, 74, 85, 68, 80, 86,  3, 68, 69, 82, 88, 87,  3, 86, 70,
        75, 82, 82, 79,  3, 79, 76, 73, 72, 15,  3, 86, 88, 70, 75,  3, 68, 86,
         3,  5, 55, 72, 68, 70, 75, 72, 85, 86,  5, 17,  3, 48, 92,  3, 22, 24,
         3, 92, 72, 68, 85, 86,  3, 76, 81,  3, 87, 75, 72,  3, 87, 72, 68, 70,
        75, 76, 81, 74,  3, 83, 85, 82, 73, 72, 86, 86, 76, 82, 81,  3, 79, 72,
        68, 71,  3, 80, 72,  3, 87, 82,  3, 69, 72, 79, 76, 72, 89, 72,  3, 87,
        75, 68, 87,  3, 37, 85, 82, 80, 90, 72, 79, 79,  3, 43, 76, 74, 75, 10,
        86,  3, 86, 68, 87, 76, 85, 72,  3, 76, 86,  3, 80, 88, 70, 75,  3, 70,
        79, 82, 86, 72, 85,  3, 87, 82, 

### Data Collator 
The **collate_fn** expects everything to be of the same size and will stack them together, but that won't work if the sizes are not the same => first pad all the samples to the longest sample in a batch and then stack them 

- Dataset: how we go into the hard drive and grab one sample 
- DataLoader: to grab a number of samples at once (batch_size)
- Data Collator (collate_fn): logic that the data loader uses to take those samples and stack them together 

In [ ]:
a = torch.ones(10)
b = torch.ones(8)
c = torch.ones(2)
print(a)
padded = nn.utils.rnn.pad_sequence([a,b,c], padding_value=0)
print(padded.shape) # [10, 3]: Sequence First : [seq length, batch size]
padded

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])
torch.Size([10, 3])


tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 0.],
        [1., 1., 0.],
        [1., 1., 0.],
        [1., 1., 0.],
        [1., 1., 0.],
        [1., 1., 0.],
        [1., 0., 0.],
        [1., 0., 0.]])

When it comes to PyTorch, we can have our data tensors in two formats: 
- Sequence First => shape: [sequence length x batch x ....]
- Batch First => shape: [batch x sequence length]

In [68]:
padded = nn.utils.rnn.pad_sequence([a,b,c], padding_value=0, batch_first=True)
print(padded.shape) # [3, 10]: Batch First : [batch size, seq length]
padded

torch.Size([3, 10])


tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [71]:
def data_collator(batch): 
    texts, labels = [], []

    for txt, lbl in batch: 
        texts.append(txt)
        labels.append(lbl)

    labels = torch.tensor(labels)
    texts = nn.utils.rnn.pad_sequence(texts, padding_value=char2idx["<PAD>"], batch_first=True)

    return texts, labels

### Run with/without Data Collator 

In [ ]:
# dataloader = DataLoader(dataset, batch_size=4) # RuntimeError: stack expects each tensor to be equal size, but got [806] at entry 0 and [2366] at entry 1
dataloader = DataLoader(dataset, batch_size=4, collate_fn=data_collator) # RuntimeError: stack expects each tensor to be equal size, but got [806] at entry 0 and [2366] at entry 1


for texts, labels in dataloader: 
    print(texts.shape)
    print(texts)
    break

torch.Size([4, 2366])
tensor([[ 37,  85,  82,  ..., 179, 179, 179],
        [ 43,  82,  80,  ...,  85,  86,  17],
        [ 37,  85,  76,  ..., 179, 179, 179],
        [ 55,  75,  76,  ..., 179, 179, 179]])


### Increase Performance of DataLoaders 
- num_workers > 0: let more threads run on CPU => grab samples in parallel 
- pin_memory = True: 
    - batches are allocated in page-locked (pinned) CPU memory. It means that it will not be swapped to hard drive by OS, even if RAM is full
    - to help faster transfer from CPU to GPU, especially with non_blocking=True in .cuda(). In this case, GPU can copy data with DMA (Direct Memory Access). 
    - Only useful for GPU training; has no effect for CPU-only training.